# On/Off Behavior (Status)

A gas boiler with operational constraints: minimum run time, startup costs,
and a minimum load when running. An expensive backup covers demand when the
main boiler is off.

**New concepts**: Status (semi-continuous dispatch · startup costs · running costs · min uptime · min downtime · prior rates)

In [ ]:
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from fluxopt import Carrier, Converter, Effect, Flow, Port, Status, optimize

pio.renderers.default = 'notebook_connected'

## Input Data — 24 h with two demand blocks

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]

# Two peaks separated by low-demand gaps.
# The boiler's minimum load is 30 kW — during hours with demand < 30,
# the boiler cannot run (it would overproduce with no sink for excess).
demand = [
    10,
    10,
    10,
    10,
    30,
    50,  # night: below min_load, then ramp
    80,
    100,
    90,
    70,
    50,
    40,  # morning peak
    10,
    10,
    10,
    10,
    10,
    30,  # afternoon: below min_load for 5 h
    60,
    90,
    100,
    80,
    50,
    20,  # evening peak
]

fig = go.Figure(go.Bar(x=timesteps, y=demand, marker_color='#ef553b'))
fig.update_layout(
    height=250,
    margin={'l': 50, 'r': 20, 't': 10, 'b': 20},
    template='plotly_white',
    yaxis_title='kW',
)
fig

## System

```
Gas Grid ──► [gas] ──► Main Boiler η=90%  ──► [heat] ──► Building
  0.05 €/kWh            min_load=30%                ▲
                         startup=30 €               │
                         running=3 €/h       Backup Heat Supply
                         min_uptime=3 h        0.20 €/kWh (expensive)
                         min_downtime=2 h
```

The **main boiler** has operational constraints: it takes time to warm up
(`min_uptime=3`), can't restart immediately after shutdown (`min_downtime=2`),
has an ignition cost (`effects_per_startup`), and a fixed hourly overhead
while running (`effects_per_running_hour`).

When it's on, it must produce at least 30% of its capacity (`relative_minimum=0.3`).
This gives **semi-continuous** behavior: output is either 0 or in [30, 100] kW.

The **backup** is a simple expensive heat source with no constraints. The solver
balances main-boiler startup/running overhead against backup cost.

In [ ]:
max_d = max(demand)

main_heat = Flow(
    'heat',
    size=100,
    relative_minimum=0.3,
    prior_rates=[0],  # boiler was off before the horizon
    status=Status(
        min_uptime=3,
        min_downtime=2,
        effects_per_startup={'cost': 30},
        effects_per_running_hour={'cost': 3},
    ),
)

result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port(
            'gas_grid',
            imports=[Flow('gas', size=500, effects_per_flow_hour={'cost': 0.05})],
        ),
        Port(
            'backup',
            imports=[Flow('heat', size=200, effects_per_flow_hour={'cost': 0.20})],
        ),
        Port(
            'building',
            exports=[
                Flow('heat', size=max_d, fixed_relative_profile=[d / max_d for d in demand]),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'main',
            thermal_efficiency=0.90,
            fuel_flow=Flow('gas', size=200),
            thermal_flow=main_heat,
        ),
    ],
    objective_effects='cost',
)

## Results

In [ ]:
print(f'Total cost: {result.objective:.2f} EUR')

In [ ]:
times = result.flow_rates.coords['time'].values

# On/off status (binary) from the solution
on = result.solution['flow--on'].sel(flow='main(heat)')
startup = result.solution['flow--startup'].sel(flow='main(heat)')

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('Heat Output', 'Main Boiler Status', 'Hourly Cost'),
)

# Row 1: heat flows
fig.add_trace(
    go.Scatter(
        x=times,
        y=result.flow_rate('main(heat)').values,
        name='main(heat)',
        line_shape='hv',
        line_color='#636efa',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=result.flow_rate('backup(heat)').values,
        name='backup(heat)',
        line_shape='hv',
        line_color='#ef553b',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=demand,
        name='demand',
        line_shape='hv',
        line={'color': 'black', 'dash': 'dot'},
    ),
    row=1,
    col=1,
)

# Row 2: on/off status + startup markers
fig.add_trace(
    go.Scatter(
        x=times,
        y=on.values,
        name='on/off',
        line_shape='hv',
        fill='tozeroy',
        line_color='#636efa',
    ),
    row=2,
    col=1,
)
startup_times = times[startup.values > 0.5]
fig.add_trace(
    go.Scatter(
        x=startup_times,
        y=[1.05] * len(startup_times),
        mode='markers',
        marker={'symbol': 'triangle-down', 'size': 12, 'color': '#ff7f0e'},
        name='startup',
    ),
    row=2,
    col=1,
)

# Row 3: cost
cost = result.effects_temporal.sel(effect='cost')
fig.add_trace(go.Bar(x=times, y=cost.values, name='cost', showlegend=False), row=3, col=1)

fig.update_layout(
    height=550,
    margin={'l': 50, 'r': 20, 't': 30, 'b': 20},
    template='plotly_white',
)
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='on/off', row=2, col=1, range=[-0.1, 1.2])
fig.update_yaxes(title_text='EUR', row=3, col=1)
fig

## Insights

- **Two on-blocks**: the solver schedules the main boiler for two runs
  (morning and evening peaks), shutting down during the low-demand
  afternoon gap. Each restart incurs the 30 EUR startup cost.

- **Semi-continuous dispatch**: when the main boiler is on, its output is
  always >= 30 kW (30% of 100 kW capacity). It never trickles at low
  output — it either runs above minimum load or shuts down entirely.

- **Minimum uptime (3 h)**: once started, the boiler commits to at least
  3 consecutive hours. The solver only starts the boiler when enough
  high-demand hours follow to justify the commitment.

- **Minimum downtime (2 h)**: after shutdown, the boiler stays off for
  at least 2 hours. During these forced-off windows, the expensive
  backup covers demand.

- **Prior rates**: `prior_rates=[0]` tells the solver the boiler was off
  before the horizon. Without this, the initial on/off state would be
  free and the solver might assume it was already running (avoiding a
  startup cost at t=0).

- **Backup fills the gaps**: the expensive backup handles all demand
  during the night and afternoon when the main boiler is off. The cost
  spikes visible in the bottom panel correspond to backup-only hours.